In [ ]:
# ========== 导入 + 检查 OpenAI API Key ==========
# 本练习：用 Selenium 无头 Chrome 抓网页正文，再让 GPT 做「毒舌」风格摘要（Day 1：抓取 → LLM）
# 需要本机 Chrome；webdriver_manager 会自动下载匹配的 ChromeDriver

# 标准库 os：读环境变量（Environment Variables），例如 OPENAI_API_KEY
import os
# OpenAI 官方 SDK：后面用 chat.completions.create 调云端模型
from openai import OpenAI
# Selenium：驱动真实浏览器抓动态/静态页面文本
from selenium import webdriver
# By：定位元素的策略（这里用 TAG_NAME 找 body）
from selenium.webdriver.common.by import By
# Chrome Options：无头、窗口大小、沙箱等启动参数
from selenium.webdriver.chrome.options import Options
# Service：指定 ChromeDriver 可执行文件路径
from selenium.webdriver.chrome.service import Service
# ChromeDriverManager：按本机 Chrome 版本自动安装/定位驱动
from webdriver_manager.chrome import ChromeDriverManager

# 若环境里没有 OPENAI_API_KEY，打印致命错误（文案保持英文）
if not os.getenv("OPENAI_API_KEY"):
    print("FATAL ERROR: OPENAI_API_KEY environment variable is not set.")
else:
    # 找到密钥后再创建客户端；OpenAI() 默认从环境变量读 KEY
    print("OpenAI API key found.")
    client = OpenAI()


In [ ]:
# ========== Prompt 模板：system 定毒舌风格；user 前缀后面拼网页正文 ==========
# 发给模型的 prompt 字符串保持英文，改译会改变回答风格/行为

system_prompt = """You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

WEBSITE CONTENT:
"""


In [ ]:
# ========== Selenium 抓取：无头 Chrome 打开 URL，返回 body 纯文本 ==========

def fetch_website_contents(url):
    # 进度日志保持英文（原代码 print 文案不翻译）
    print(f"Fetching content from {url} using Selenium...")
    # driver 先置空，方便 finally 里判断是否需要关闭
    driver = None
    
    try:
        print("1. Setting up Chrome options...")
        # 创建 Chrome 启动选项对象
        chrome_options = Options()
        # 无头模式：不弹出可见浏览器窗口
        chrome_options.add_argument("--headless")
        # 部分环境下关闭 GPU 加速更稳
        chrome_options.add_argument("--disable-gpu")
        # 设定视口大小，影响部分响应式页面渲染
        chrome_options.add_argument("window-size=1200x600")
        # 在容器/CI 里常需 --no-sandbox
        chrome_options.add_argument("--no-sandbox")
        # 避免 /dev/shm 过小导致 Chrome 崩溃
        chrome_options.add_argument("--disable-dev-shm-usage")

        print("2. Installing/finding WebDriver...")
        # 自动下载或复用匹配的 ChromeDriver，返回可执行路径
        driver_path = ChromeDriverManager().install()
        print(f"WebDriver is at: {driver_path}")
        # 用该路径构造 Service
        service = Service(driver_path)
        
        print("3. Initializing WebDriver...")
        # 启动 Chrome 实例
        driver = webdriver.Chrome(service=service, options=chrome_options)
        
        print("4. Setting page load timeout...")
        # 整页加载最多等 10 秒，超时抛错（下面 except 可能仍取部分正文）
        driver.set_page_load_timeout(10)

        print(f"5. Getting URL: {url}...")
        # 导航到目标网址
        driver.get(url)
        
        print("6. Setting implicit wait...")
        # 隐式等待：找元素时最多再等 5 秒
        driver.implicitly_wait(5)
        
        print("7. Finding body content...")
        # 取 <body> 的可见文本（不是 HTML 源码）
        body_content = driver.find_element(By.TAG_NAME, "body").text
        
        print("Content fetched successfully.")
        return body_content
    
    except Exception as e:
        # 超时或其它 Selenium 错误：尝试仍读 body，有内容就当部分成功
        print(f"Error during Selenium operation: {e}")
        if driver:
            body_content = driver.find_element(By.TAG_NAME, "body").text
            if body_content:
                print("Page timed out, but returning partial content.")
                return body_content
        # 实在拿不到就返回 None，上层 summarize 会友好提示
        return None
    finally:
        # 无论成功失败都关闭浏览器，避免残留进程
        if driver:
            print("8. Shutting down WebDriver...")
            driver.close()
            driver.quit()
            print("WebDriver shut down.")


In [ ]:
# ========== 摘要入口：抓网页 → 组 messages → 调 GPT → 返回 Markdown 文本 ==========

def summarize(url):
    # 先用 Selenium 拉正文
    website_content = fetch_website_contents(url)
    
    # 抓取失败：返回固定英文道歉句（影响行为的字符串不翻译）
    if not website_content:
        return "Sorry, I couldn't fetch the website. It's probably broken or my robot eyes can't see it."

    # system 定毒舌风格；user = 前缀 + 网页正文
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website_content}
    ]

    try:
        print("9. Calling OpenAI API...")
        # Chat Completions：模型名与参数保持原样
        response = client.chat.completions.create(
            model="gpt-4-turbo",
            messages=messages
        )
        print("OpenAI call successful.")
        # 取第一条 choice 的助手回复正文
        return response.choices[0].message.content
    
    except Exception as e:
        # API 失败时打印错误并返回英文占位句
        print(f"Error calling OpenAI: {e}")
        return "Error from AI: It probably got bored and left."


In [ ]:
# ========== 试跑：改 target_url 就能摘要别的网站 ==========

# 目标网址（示例站）；换成你想分析的 URL
target_url = "https://example.com"
# 调用 summarize：内部会 Selenium 抓取 + OpenAI 摘要
snarky_summary = summarize(target_url)

# 打印分隔标题与最终摘要（英文标签保留）
print("\n--- SNARKY SUMMARY ---")
print(snarky_summary)
